# 06 · Handling Imbalanced Data

        > Part of the **Data Preprocessing** module — taught alongside the `data/customer_churn.csv` dataset.

        ## Learning objectives
        - See why accuracy lies on imbalanced problems — and what to use instead
- Apply random oversampling, undersampling, and SMOTE / SMOTENC / ADASYN
- Use `class_weight` as a one-line alternative to resampling
- Combine over- and under-sampling (SMOTEENN, SMOTETomek)
- Resample **only on the training fold** — the most common bug in this area

        ---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

df = pd.read_csv("data/customer_churn.csv")
print("shape:", df.shape)
df.head()

> Requires the `imbalanced-learn` package.
> `pip install imbalanced-learn` if you haven't already.

In [ ]:
df["churned"].value_counts(normalize=True).round(3)

## 1. Why accuracy is misleading

Predict "no churn" for everyone on a 9% churn dataset → 91% accuracy. Useless model.

What to use instead:
- **Confusion matrix** — see TP/FP/TN/FN directly
- **Precision** — of those we flagged, how many were right
- **Recall (sensitivity)** — of the real churners, how many did we catch
- **F1** — harmonic mean of precision and recall
- **ROC-AUC** — overall ranking quality
- **PR-AUC** — preferred over ROC-AUC when the positive class is rare

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, average_precision_score)

X = df.drop(columns=["customer_id", "churned"])
y = df["churned"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
     cat_cols),
])
X_train_p = pre.fit_transform(X_train)
X_test_p  = pre.transform(X_test)

def evaluate(name, model):
    proba = model.predict_proba(X_test_p)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    print(f"--- {name} ---")
    print(confusion_matrix(y_test, pred))
    print(f"ROC-AUC: {roc_auc_score(y_test, proba):.3f}   "
          f"PR-AUC: {average_precision_score(y_test, proba):.3f}")
    print(classification_report(y_test, pred, digits=3))

baseline = LogisticRegression(max_iter=1000).fit(X_train_p, y_train)
evaluate("Baseline (no resampling)", baseline)

## 2. The cheapest fix — `class_weight="balanced"`

The model penalises mistakes on the minority class more during training. Often as good
as resampling, with no extra plumbing.

In [ ]:
weighted = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_train_p, y_train)
evaluate("class_weight='balanced'", weighted)

## 3. Random Oversampling

Duplicate minority class samples until the classes are balanced. Risk: the same rows
appear many times → models can memorise them and overfit.

In [ ]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_ros, y_ros = ros.fit_resample(X_train_p, y_train)
print("After ROS:", np.bincount(y_ros))
evaluate("Random Oversampling", LogisticRegression(max_iter=1000).fit(X_ros, y_ros))

## 4. SMOTE — Synthetic Minority Oversampling

Instead of duplicating, SMOTE *creates new* minority samples by interpolating between
existing ones and their nearest neighbours. The default modern choice for tabular
imbalance.

⚠ SMOTE works on **numeric** features. For mixed numeric+categorical, use **SMOTENC**.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42, k_neighbors=5)
X_sm, y_sm = smote.fit_resample(X_train_p, y_train)
print("After SMOTE:", np.bincount(y_sm))
evaluate("SMOTE", LogisticRegression(max_iter=1000).fit(X_sm, y_sm))

## 5. ADASYN

Like SMOTE but generates more synthetic samples *near hard-to-classify* minority points
(those near the decision boundary). Sometimes helps, sometimes hurts.

In [ ]:
from imblearn.over_sampling import ADASYN

ada = ADASYN(random_state=42)
X_ad, y_ad = ada.fit_resample(X_train_p, y_train)
evaluate("ADASYN", LogisticRegression(max_iter=1000).fit(X_ad, y_ad))

## 6. Random Undersampling

Drop majority class samples until balanced. Fast, but you throw away information.
Useful when the dataset is huge and the majority class is highly redundant.

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(X_train_p, y_train)
print("After RUS:", np.bincount(y_rus))
evaluate("Random Undersampling", LogisticRegression(max_iter=1000).fit(X_rus, y_rus))

## 7. Tomek Links and SMOTEENN

- **Tomek links** remove ambiguous pairs at the class boundary → cleaner separation.
- **SMOTEENN** = SMOTE + Edited Nearest Neighbours: oversamples the minority then
  cleans up noisy synthetic points. Often the best single recipe.

In [ ]:
from imblearn.combine import SMOTEENN

smen = SMOTEENN(random_state=42)
X_se, y_se = smen.fit_resample(X_train_p, y_train)
print("After SMOTEENN:", np.bincount(y_se))
evaluate("SMOTEENN", LogisticRegression(max_iter=1000).fit(X_se, y_se))

## 8. The critical rule — resample only on the training fold

Resampling the whole dataset before splitting **leaks** synthetic / duplicated samples
into the test set → optimistic, useless metrics.

The right way is `imblearn.pipeline.Pipeline` (note: `imblearn`'s pipeline, not
`sklearn`'s — only the imblearn one is aware of resamplers and runs them only on training data).

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

pipe = ImbPipeline([
    ("pre",   pre),
    ("smote", SMOTE(random_state=42)),
    ("clf",   LogisticRegression(max_iter=1000)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring="average_precision", n_jobs=-1)
print(f"PR-AUC (CV): {scores.mean():.3f}  ± {scores.std():.3f}")

## 9. Threshold tuning — often more impactful than resampling

The default threshold of 0.5 is rarely optimal on imbalanced data. Sweep thresholds
and pick one that matches your business cost (e.g. you care about recall ≥ 0.8).

In [ ]:
from sklearn.metrics import precision_recall_curve

proba = baseline.predict_proba(X_test_p)[:, 1]
prec, rec, thr = precision_recall_curve(y_test, proba)

plt.figure(figsize=(7, 4))
plt.plot(thr, prec[:-1], label="precision")
plt.plot(thr, rec[:-1],  label="recall")
plt.xlabel("threshold"); plt.ylabel("score"); plt.legend(); plt.title("Precision–Recall vs threshold")
plt.show()

## 10. Cheat sheet

| Situation | First thing to try |
|-----------|---------------------|
| Slight imbalance (30/70) | nothing — just use proper metrics |
| Moderate imbalance | `class_weight="balanced"` |
| Strong imbalance, numeric features | SMOTE |
| Strong imbalance, mixed features | SMOTENC |
| Huge dataset, redundant majority | random undersampling |
| Noisy boundary | SMOTEENN / SMOTETomek |
| Always | tune the decision threshold |

## Exercise
1. Repeat the comparison with a `RandomForestClassifier`. Does SMOTE still help, or
   does `class_weight="balanced_subsample"` already do the job?
2. Replace the metric in `cross_val_score` with `f1`. Does the ranking of methods change?